In [1]:
import json
import csv
from pathlib import Path

# Uses the exact output folder from your previous notebook
OUTPUT_DIR = Path(r"I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit")

# Load the 3 baseline answer files
with open(OUTPUT_DIR / "english_base_answer.json", "r", encoding="utf-8") as f:
    en_data = json.load(f)

with open(OUTPUT_DIR / "bangla_base_answer.json", "r", encoding="utf-8") as f:
    bn_data = json.load(f)

with open(OUTPUT_DIR / "banglish_base_answer.json", "r", encoding="utf-8") as f:
    bl_data = json.load(f)

# Save them side-by-side into a CSV file
# utf-8-sig ensures Bangla text displays properly if you open it in Excel
csv_path = OUTPUT_DIR / "baseline_review.csv"

with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow([
        "id",
        "en_question", "en_answer", "en_correct",
        "bn_question", "bn_answer", "bn_correct",
        "bl_question", "bl_answer", "bl_correct"
    ])

    for i in range(len(en_data)):
        writer.writerow([
            i + 1,
            en_data[i]["question"], en_data[i]["answer"].replace("\n", " "), "",
            bn_data[i]["question"], bn_data[i]["answer"].replace("\n", " "), "",
            bl_data[i]["question"], bl_data[i]["answer"].replace("\n", " "), ""
        ])

print(f"Done! Open this file to grade: {csv_path}")

Done! Open this file to grade: I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit\baseline_review.csv


In [2]:
import json
import csv
from pathlib import Path

OUTPUT_DIR = Path(r"I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit")
csv_path = OUTPUT_DIR / "baseline_review.csv"

en_score, bn_score, bl_score = 0, 0, 0
total = 0
verified_facts = []

with open(csv_path, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        total += 1
        en_ok = int(row["en_correct"])
        bn_ok = int(row["bn_correct"])
        bl_ok = int(row["bl_correct"])

        en_score += en_ok
        bn_score += bn_ok
        bl_score += bl_ok

        # Keep facts the model got right in BOTH English and Bangla
        # (Change to `if en_ok == 1 and bn_ok == 1 and bl_ok == 1:` if you want all 3)
        if en_ok == 1 and bn_ok == 1:
            verified_facts.append({
                "id": int(row["id"]),
                "en_question": row["en_question"],
                "bn_question": row["bn_question"],
                "bl_question": row["bl_question"],
                "base_en_answer": row["en_answer"],
                "base_bn_answer": row["bn_answer"],
                "base_bl_answer": row["bl_answer"],
                "bl_known": bl_ok
            })

print(f"Total Questions: {total}")
print(f"English Accuracy:  {en_score}/{total} ({en_score/total*100:.1f}%)")
print(f"Bangla Accuracy:   {bn_score}/{total} ({bn_score/total*100:.1f}%)")
print(f"Banglish Accuracy: {bl_score}/{total} ({bl_score/total*100:.1f}%)")
print(f"\nVerified facts ready for Step 2: {len(verified_facts)}")

save_path = OUTPUT_DIR / "verified_known_facts.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(verified_facts, f, ensure_ascii=False, indent=4)

print(f"Saved clean dataset to: {save_path}")

Total Questions: 50
English Accuracy:  47/50 (94.0%)
Bangla Accuracy:   39/50 (78.0%)
Banglish Accuracy: 15/50 (30.0%)

Verified facts ready for Step 2: 38
Saved clean dataset to: I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit\verified_known_facts.json
